## Import Packages


In [1]:
%pip install PyWavelets


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
import pandas as pd
import numpy as np
import scipy.stats as sp
import pywt

.

.

.


## Declare the size of feature dataset


In [17]:
NoOfData = (
    180  # 180 Data for each robotic spot-welding condition (Normal, Abnormal)
)
NoOfSensor = 3  # 3 Sensor signals: Acceleration, Voltage, Current
NoOfFeature = 10  # 10 Feature types: Max, Min, Mean, RMS, Variance, Skewness, Kurtosis, Crest factor, Shape factor, Impulse factor

NoOfData, NoOfSensor, NoOfFeature

(180, 3, 10)

## Load Raw Dataset (360 files)


In [18]:
for i in range(NoOfData):

    temp_path1 = f"https://github.com/purduelamm/purdue_me597_iiot/blob/main/ml_tutorial/Dataset/Normal_{i+1}?raw=true"  # File path of temporary normal data
    temp_path2 = f"https://github.com/purduelamm/purdue_me597_iiot/blob/main/ml_tutorial/Dataset/Abnormal_{i+1}?raw=true"  # File path of temporary abnormal data

    exec(f"Normal_{i+1}   = pd.read_csv(temp_path1 , sep=',' , header=None)")
    exec(f"Abnormal_{i+1} = pd.read_csv(temp_path2 , sep=',' , header=None)")

## Time Domain Feature Extraction

- 10 features \* 3 sensors = 30 features


In [19]:
# Definition of rms function
def rms(x):
    return np.sqrt(np.mean(x**2))

In [20]:
# Create empty(0) arrays for normal/abnormal feature dataset (time domain)
TimeFeature_Normal = np.zeros((NoOfSensor * NoOfFeature, NoOfData))
TimeFeature_Abnormal = np.zeros((NoOfSensor * NoOfFeature, NoOfData))

print(TimeFeature_Normal.shape)
print(TimeFeature_Abnormal.shape)

TimeFeature_Normal

(30, 180)
(30, 180)


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(30, 180))

In [21]:
for i in range(NoOfData):

    # Declare temporary data
    exec(f"temp_data1 = Normal_{i+1}")
    exec(f"temp_data2 = Abnormal_{i+1}")

    # Time domain feature extraction
    for j in range(NoOfSensor):

        # Normal features
        TimeFeature_Normal[NoOfFeature * j + 0, i] = np.max(
            temp_data1.iloc[:, j + 1]
        )
        TimeFeature_Normal[NoOfFeature * j + 1, i] = np.min(
            temp_data1.iloc[:, j + 1]
        )
        TimeFeature_Normal[NoOfFeature * j + 2, i] = np.mean(
            temp_data1.iloc[:, j + 1]
        )
        TimeFeature_Normal[NoOfFeature * j + 3, i] = rms(
            temp_data1.iloc[:, j + 1]
        )
        TimeFeature_Normal[NoOfFeature * j + 4, i] = np.var(
            temp_data1.iloc[:, j + 1]
        )
        TimeFeature_Normal[NoOfFeature * j + 5, i] = sp.skew(
            temp_data1.iloc[:, j + 1]
        )
        TimeFeature_Normal[NoOfFeature * j + 6, i] = sp.kurtosis(
            temp_data1.iloc[:, j + 1]
        )
        TimeFeature_Normal[NoOfFeature * j + 7, i] = np.max(
            temp_data1.iloc[:, j + 1]
        ) / rms(temp_data1.iloc[:, j + 1])
        TimeFeature_Normal[NoOfFeature * j + 8, i] = rms(
            temp_data1.iloc[:, j + 1]
        ) / np.mean(np.abs(temp_data1.iloc[:, j + 1]))
        TimeFeature_Normal[NoOfFeature * j + 9, i] = np.max(
            temp_data1.iloc[:, j + 1]
        ) / np.mean(np.abs(temp_data1.iloc[:, j + 1]))

        # Abnormal features
        TimeFeature_Abnormal[NoOfFeature * j + 0, i] = np.max(
            temp_data2.iloc[:, j + 1]
        )
        TimeFeature_Abnormal[NoOfFeature * j + 1, i] = np.min(
            temp_data2.iloc[:, j + 1]
        )
        TimeFeature_Abnormal[NoOfFeature * j + 2, i] = np.mean(
            temp_data2.iloc[:, j + 1]
        )
        TimeFeature_Abnormal[NoOfFeature * j + 3, i] = rms(
            temp_data2.iloc[:, j + 1]
        )
        TimeFeature_Abnormal[NoOfFeature * j + 4, i] = np.var(
            temp_data2.iloc[:, j + 1]
        )
        TimeFeature_Abnormal[NoOfFeature * j + 5, i] = sp.skew(
            temp_data2.iloc[:, j + 1]
        )
        TimeFeature_Abnormal[NoOfFeature * j + 6, i] = sp.kurtosis(
            temp_data2.iloc[:, j + 1]
        )
        TimeFeature_Abnormal[NoOfFeature * j + 7, i] = np.max(
            temp_data2.iloc[:, j + 1]
        ) / rms(temp_data2.iloc[:, j + 1])
        TimeFeature_Abnormal[NoOfFeature * j + 8, i] = rms(
            temp_data2.iloc[:, j + 1]
        ) / np.mean(np.abs(temp_data2.iloc[:, j + 1]))
        TimeFeature_Abnormal[NoOfFeature * j + 9, i] = np.max(
            temp_data2.iloc[:, j + 1]
        ) / np.mean(np.abs(temp_data2.iloc[:, j + 1]))

print(TimeFeature_Normal.shape)
print(TimeFeature_Abnormal.shape)

TimeFeature_Normal

(30, 180)
(30, 180)


array([[ 1.35100000e+00,  3.16610000e+01,  3.18320000e+01, ...,
         1.05340000e+00,  1.08740000e+00,  9.58390000e-01],
       [-1.37200000e+00, -2.27860000e+01, -2.36130000e+01, ...,
        -9.65520000e-01, -1.16450000e+00, -8.99710000e-01],
       [ 1.10828100e-02,  2.33385723e-02,  2.05055037e-02, ...,
         3.09079384e-02,  3.20263227e-02,  3.37205056e-02],
       ...,
       [ 1.90403829e+00,  1.93966077e+00,  1.94395765e+00, ...,
         1.89501956e+00,  1.91625999e+00,  1.90413317e+00],
       [ 1.44617765e+00,  1.44713933e+00,  1.44831636e+00, ...,
         1.44636143e+00,  1.44619809e+00,  1.44548575e+00],
       [ 2.75357761e+00,  2.80695940e+00,  2.81546567e+00, ...,
         2.74088319e+00,  2.77129154e+00,  2.75239737e+00]],
      shape=(30, 180))

### Combine Normal and Abnormal feature arrays

- axis=0: combine rows
- axis=1: combine columns


In [22]:
TimeFeature = np.concatenate(
    [TimeFeature_Normal, TimeFeature_Abnormal], axis=1
)
TimeFeature.shape

(30, 360)

.

.

.


## Frequency Domain Feature Extraction

- 10 features _ 8 wavelet levels _ 3 sensors = 240 features


In [23]:
# Wavelet options
MotherWavelet = pywt.Wavelet("haar")  # Mother wavelet
Level = 8  # Wavelet decomposition level

In [24]:
# Create empty(0) arrays for normal/abnormal feature dataset (frequency Domain)
FreqFeature_Normal = np.zeros(
    shape=(NoOfSensor * NoOfFeature * Level, NoOfData)
)
FreqFeature_Abnormal = np.zeros(
    shape=(NoOfSensor * NoOfFeature * Level, NoOfData)
)

print(FreqFeature_Normal.shape)
print(FreqFeature_Abnormal.shape)

FreqFeature_Normal

(240, 180)
(240, 180)


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(240, 180))

In [25]:
for i in range(NoOfData):

    # Declare temporary data (only sensor signals)
    exec(f"temp_data1 = Normal_{i+1}.iloc[:,1:]")
    exec(f"temp_data2 = Abnormal_{i+1}.iloc[:,1:]")

    # Walvelet decomposition
    Coef1 = pywt.wavedec(temp_data1, MotherWavelet, level=Level, axis=0)
    Coef2 = pywt.wavedec(temp_data2, MotherWavelet, level=Level, axis=0)

    # Frequency domain feature extraction
    for j in range(NoOfSensor):

        for k in np.arange(Level):
            coef1 = Coef1[Level-k]
            coef2 = Coef2[Level-k]

            ##################################################
            # Complete code below to obtain proper features
            # Tip: Use NoOfFeature, Level, j, and k
            ##################################################

            # Normal features
            FreqFeature_Normal[ , i] = np.max(coef1[:,j])
            FreqFeature_Normal[ , i] = np.min(coef1[:,j])
            FreqFeature_Normal[ , i] = np.mean(coef1[:,j])
            FreqFeature_Normal[ , i] = rms(coef1[:,j])
            FreqFeature_Normal[ , i] = np.var(coef1[:,j])
            FreqFeature_Normal[ , i] = sp.skew(coef1[:,j])
            FreqFeature_Normal[ , i] = sp.kurtosis(coef1[:,j])
            FreqFeature_Normal[ , i] = np.max(coef1[:,j])/rms(coef1[:,j])
            FreqFeature_Normal[ , i] = rms(coef1[:,j])/np.mean(np.abs(coef1[:,j]))
            FreqFeature_Normal[ , i] = np.max(coef1[:,j])/np.mean(np.abs(coef1[:,j]))

            # Abnormal features
            FreqFeature_Abnormal[ , i] = np.max(coef2[:,j])
            FreqFeature_Abnormal[ , i] = np.min(coef2[:,j])
            FreqFeature_Abnormal[ , i] = np.mean(coef2[:,j])
            FreqFeature_Abnormal[ , i] = rms(coef2[:,j])
            FreqFeature_Abnormal[ , i] = np.var(coef2[:,j])
            FreqFeature_Abnormal[ , i] = sp.skew(coef2[:,j])
            FreqFeature_Abnormal[ , i] = sp.kurtosis(coef2[:,j])
            FreqFeature_Abnormal[ , i] = np.max(coef2[:,j])/rms(coef2[:,j])
            FreqFeature_Abnormal[ , i] = rms(coef2[:,j])/np.mean(np.abs(coef2[:,j]))
            FreqFeature_Abnormal[ , i] = np.max(coef2[:,j])/np.mean(np.abs(coef2[:,j]))

            ##################################################
            ##################################################

print(FreqFeature_Normal.shape)
print(FreqFeature_Abnormal.shape)

FreqFeature_Normal

SyntaxError: invalid syntax (3206690358.py, line 24)

### Combine Normal and Abnormal feature arrays

- axis=0: combine rows
- axis=1: combine columns


In [ ]:
FreqFeature = np.concatenate(
    [FreqFeature_Normal, FreqFeature_Abnormal], axis=1
)
FreqFeature.shape

.

.

.


## Final Feature Dataset

- (30 Time domain features + 240 Frequency domain features = 270 features)


In [ ]:
Features = np.concatenate([TimeFeature, FreqFeature], axis=0)

print(Features.shape)
Features

### Convert Array into Data frame format

- Easy to save as data file (csv)


In [ ]:
Features_df = pd.DataFrame(Features)
Features_df

### Save Final Feature Data in .csv


In [ ]:
path = "./data/FeatureData.csv"
Features_df.to_csv(path, sep=",", header=None, index=None)